# MNIST Veri Seti ile MLP Tabanlı El Yazısı Rakam Tanıma
## Derin Öğrenme Dersi Proje Ödevi

**Ödev Amacı:**
- MNIST veri seti üzerinde bir **Multi-Layer Perceptron (MLP)** modeli geliştirmek, eğitmek ve test etmek.
- **Öğrenci Okul Numarası** rakamlarını kendi el yazımızla kağıda yazarak modele vermek ve modelin gerçek dünya verisindeki başarımını (**Distribution Shift / Generalization**) değerlendirmek.
- Farklı optimizasyon ve katman mimarilerini karşılaştırmak.

> **Önemli Kurallar:**
> 1. **CNN KESİNLİKLE KULLANILMAMIŞTIR.** Sadece Fully Connected (MLP) katmanlar kullanılmıştır.
> 2. Hazır eğitilmiş model kullanılmamış, modeller **sıfırdan eğitilmiştir**.
> 3. **Train / Validation / Test** ayrımı yapılmış, test kümesine eğitim süresince dokunulmamıştır.

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from PIL import Image

# Proje kaynak modüllerini ekle
sys.path.append('..')
from src.dataset import get_data_loaders, get_mnist_transforms
from src.models import BaseMLP, DeeperMLP, count_parameters
from src.preprocess_custom import preprocess_single_digit, preprocess_all_custom_images
from src.evaluate_custom import evaluate_custom_digits, print_distribution_shift_analysis

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Kullanılan Cihaz: {device}')

## 1. Veri Seti Yükleme ve Train / Validation / Test Ayrımı
- Toplam 60.000 eğitim örneği **50.000 Train** ve **10.000 Validation** olarak bölünmüştür.
- **10.000 Test** örneği tamamen izole edilmiştir.

In [ ]:
train_loader, val_loader, test_loader, (train_ds, val_ds, test_ds) = get_data_loaders(data_dir='../data', batch_size=64)

# Örnek MNIST Görüntülerini Görselleştir
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(f'Etiket: {labels[i].item()}')
    ax.axis('off')
plt.suptitle('MNIST Eğitim Kümesinden Örnekler', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Deney 1: Temel MLP Modeli Mimarisi
- Girdi: 784 (28x28 piksel)
- 1. Gizli Katman: 512 nöron + ReLU + Dropout(0.2)
- 2. Gizli Katman: 256 nöron + ReLU + Dropout(0.2)
- Çıktı: 10 nöron (0-9 sınıfları)

In [ ]:
model = BaseMLP(hidden_dims=[512, 256], dropout_rate=0.2).to(device)
print(model)
print(f'Toplam Eğitilebilir Parametre Sayısı: {count_parameters(model):,}')

## 3. Model Eğitimi ve Doğrulama Eğrileri
Eğitilmiş model ağırlıkları `models/Deney1_BaseMLP_best.pth` dosyasından yüklenmekte ya da sıfırdan eğitilebilmektedir.

In [ ]:
# Kaydedilmiş eğitim grafiğini göster
curve_img = Image.open('../figures/Deney1_BaseMLP_curves.png')
plt.figure(figsize=(14, 6))
plt.imshow(curve_img)
plt.axis('off')
plt.title('Deney 1 (Temel MLP) Eğitim ve Doğrulama Kayıp / Doğruluk Eğrileri')
plt.show()

## 4. Test Kümesi Değerlendirmesi ve Karmaşıklık Matrisi (Confusion Matrix)

In [ ]:
cm_img = Image.open('../figures/confusion_matrix.png')
plt.figure(figsize=(9, 8))
plt.imshow(cm_img)
plt.axis('off')
plt.title('MNIST Test Kümesi (10.000 Örnek) Karmaşıklık Matrisi')
plt.show()

## 5. Kendi El Yazısı Rakamlarımızla Test (Öğrenci Numarası)
Gerçek hayatta beyaz kağıda el yazısıyla yazılmış rakamlar:
1. Gri seviyeye dönüştürülür.
2. Renkler tersine çevrilir (Kağıt beyaz/yazı siyah -> MNIST arka plan siyah/yazı beyaz).
3. 20x20 boyuta en-boy oranı korunarak sığdırılır.
4. 28x28 siyah tuvalin kütle merkezine yerleştirilir ve normalize edilir.

In [ ]:
eval_res = evaluate_custom_digits(
    model_checkpoint_path='../models/Deney1_BaseMLP_best.pth',
    custom_dir='../data/custom_digits/processed',
    figures_dir='../figures'
)
pred_img = Image.open('../figures/custom_digit_predictions.png')
plt.figure(figsize=(14, 4))
plt.imshow(pred_img)
plt.axis('off')
plt.show()

print_distribution_shift_analysis(eval_res, test_acc=98.15)

## 6. Model Yapılandırmalarını Karşılaştırma Deneyleri
- **Deney 1:** Temel MLP (2 Gizli Katman, Adam lr=0.001)
- **Deney 2:** SGD Optimizer (2 Gizli Katman, SGD+Momentum lr=0.01)
- **Deney 3:** Daha Derin MLP (3 Gizli Katman: 512-256-128, Adam lr=0.001)

In [ ]:
comp_img = Image.open('../figures/all_experiments_comparison.png')
plt.figure(figsize=(15, 6))
plt.imshow(comp_img)
plt.axis('off')
plt.title('Tüm Deneylerin Karşılaştırma Grafiği')
plt.show()